# Batch Conditioning Example with Ax

This notebook demonstrates the **recommended approach** for batch conditioning (also known as pending observations or fantasy modeling) in Ax v1.

**Key Points:**
- Use `get_next_trials()` for batch generation with automatic batch conditioning
- Ax automatically tracks trials in RUNNING state as pending observations
- No manual fantasy point updates needed!

## 1. Setup and Imports

In [ ]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
from ax.modelbridge.generation_strategy import GenerationStrategy, GenerationStep
from ax.modelbridge.factory import Generators
import numpy as np

## 2. Create Experiment with Parallelism Limits

We'll create a simple optimization experiment with a GenerationStrategy that specifies parallelism limits.

In [ ]:
# Define generation strategy with parallelism limits
gs = GenerationStrategy(
    steps=[
        GenerationStep(
            model=Generators.SOBOL,
            num_trials=5,
            max_parallelism=3,  # Can run up to 3 Sobol trials in parallel
        ),
        GenerationStep(
            model=Generators.BOTORCH_MODULAR,
            num_trials=-1,  # Unlimited trials
            max_parallelism=2,  # Can run up to 2 BO trials in parallel
        ),
    ]
)

# Create AxClient with the generation strategy
ax_client = AxClient(generation_strategy=gs, verbose_logging=True)

# Create a simple 2D optimization problem
ax_client.create_experiment(
    name="batch_conditioning_example",
    parameters=[
        {"name": "x1", "type": "range", "bounds": [-5.0, 10.0]},
        {"name": "x2", "type": "range", "bounds": [0.0, 15.0]},
    ],
    objectives={"objective": ObjectiveProperties(minimize=True)},
)

print("Experiment created successfully!")

## 3. Define Evaluation Function

For this example, we'll use a simple synthetic function: Branin.

In [ ]:
def branin(x1, x2):
    """Branin test function (2D)."""
    a = 1.0
    b = 5.1 / (4.0 * np.pi ** 2)
    c = 5.0 / np.pi
    r = 6.0
    s = 10.0
    t = 1.0 / (8.0 * np.pi)
    
    result = a * (x2 - b * x1 ** 2 + c * x1 - r) ** 2 + s * (1 - t) * np.cos(x1) + s
    # Add some noise
    noise = np.random.normal(0, 0.1)
    return {"objective": (result + noise, 0.1)}

## 4. Batch Generation with Automatic Batch Conditioning

### Recommended Approach: Use `get_next_trials`

This method automatically handles batch conditioning by:
1. Creating trials in RUNNING state
2. Treating these trials as pending observations for subsequent generations
3. Respecting parallelism limits from the GenerationStrategy

In [ ]:
# Generate a batch of trials
# The method will automatically limit batch size based on parallelism settings
batch_size = 5  # Request 5 trials

trials_dict, optimization_complete = ax_client.get_next_trials(max_trials=batch_size)

print(f"\nGenerated {len(trials_dict)} trials (requested {batch_size})")
print(f"Optimization complete: {optimization_complete}\n")

# Display the generated trials
for trial_idx, params in trials_dict.items():
    print(f"Trial {trial_idx}: x1={params['x1']:.4f}, x2={params['x2']:.4f}")

# Check trial status - they should all be RUNNING
print("\nTrial statuses:")
for trial_idx in trials_dict.keys():
    trial = ax_client.experiment.trials[trial_idx]
    print(f"Trial {trial_idx}: {trial.status}")

### Key Observation

Notice that:
- We requested 5 trials, but only got 3 (respecting max_parallelism=3 for Sobol)
- All trials are in RUNNING state
- These trials will automatically be treated as pending observations in the next batch

## 5. Generate Another Batch (Shows Batch Conditioning)

Let's generate another batch **without completing the first batch**. Ax will automatically condition on the pending trials.

In [ ]:
# Try to generate more trials - should return 0 because we've hit parallelism limit
# and haven't completed any trials yet
trials_dict_2, optimization_complete = ax_client.get_next_trials(max_trials=5)

print(f"Generated {len(trials_dict_2)} additional trials")
print(f"This is expected: we can't generate more until we complete some trials.\n")

# Check current generation limits
num_trials_available, is_complete = ax_client.get_current_trial_generation_limit()
print(f"Current generation limit: {num_trials_available} trials")
print(f"Optimization complete: {is_complete}")

## 6. Complete Trials and Continue

Now let's complete some trials and generate more.

In [ ]:
# Complete the first batch of trials
for trial_idx, params in trials_dict.items():
    # Evaluate the trial
    results = branin(params['x1'], params['x2'])
    
    # Complete the trial with actual data
    ax_client.complete_trial(trial_index=trial_idx, raw_data=results)
    print(f"Completed trial {trial_idx} with objective = {results['objective'][0]:.4f}")

In [ ]:
# Now generate more trials - should get the remaining Sobol trials
trials_dict_3, optimization_complete = ax_client.get_next_trials(max_trials=5)

print(f"\nGenerated {len(trials_dict_3)} more trials")
print(f"Total trials so far: {len(ax_client.experiment.trials)}\n")

# Display the new trials
for trial_idx, params in trials_dict_3.items():
    print(f"Trial {trial_idx}: x1={params['x1']:.4f}, x2={params['x2']:.4f}")

## 7. Complete Remaining Trials and Move to Bayesian Optimization

Let's complete all Sobol trials and move to the BO phase.

In [ ]:
# Complete the remaining Sobol trials
for trial_idx, params in trials_dict_3.items():
    results = branin(params['x1'], params['x2'])
    ax_client.complete_trial(trial_index=trial_idx, raw_data=results)
    print(f"Completed trial {trial_idx}")

print(f"\nCompleted all Sobol trials. Total: {len(ax_client.experiment.trials)} trials")

In [ ]:
# Generate BO trials - max_parallelism=2 for this phase
trials_dict_4, optimization_complete = ax_client.get_next_trials(max_trials=3)

print(f"Generated {len(trials_dict_4)} BO trials (requested 3, limited to 2 by parallelism)\n")

# Display the BO trials
for trial_idx, params in trials_dict_4.items():
    trial = ax_client.experiment.trials[trial_idx]
    print(f"Trial {trial_idx}: x1={params['x1']:.4f}, x2={params['x2']:.4f}")
    print(f"  Generated by: {trial.generator_run._model_key}")

## 8. Verify Batch Conditioning is Working

Let's verify that the BO model is actually conditioning on pending observations.

In [ ]:
# The two BO trials should be diverse (not identical)
# This shows that the second trial was generated conditional on the first being pending
trial_indices = list(trials_dict_4.keys())
if len(trial_indices) >= 2:
    params_1 = trials_dict_4[trial_indices[0]]
    params_2 = trials_dict_4[trial_indices[1]]
    
    distance = np.sqrt(
        (params_1['x1'] - params_2['x1'])**2 + 
        (params_1['x2'] - params_2['x2'])**2
    )
    
    print(f"Distance between the two BO trials: {distance:.4f}")
    print(f"\nIf batch conditioning is working correctly, these trials should be")
    print(f"reasonably separated (distance > 0.5 typically indicates good diversity).")

## 9. Complete Final Trials and View Results

In [ ]:
# Complete the BO trials
for trial_idx, params in trials_dict_4.items():
    results = branin(params['x1'], params['x2'])
    ax_client.complete_trial(trial_index=trial_idx, raw_data=results)
    print(f"Completed trial {trial_idx} with objective = {results['objective'][0]:.4f}")

In [ ]:
# View all trials
df = ax_client.get_trials_data_frame()
print("\nAll trials:")
print(df[['trial_index', 'arm_name', 'trial_status', 'generation_node', 'objective', 'x1', 'x2']])

In [ ]:
# Get best observed trial
best_params, best_values = ax_client.get_best_parameters()

print(f"\nBest parameters found:")
print(f"  x1 = {best_params['x1']:.4f}")
print(f"  x2 = {best_params['x2']:.4f}")
print(f"\nBest objective value: {best_values[0]['objective']:.4f}")

## Summary

### What We Demonstrated:

1. ✅ **Automatic batch conditioning**: Used `get_next_trials()` which automatically handles pending observations
2. ✅ **Parallelism limits**: The GenerationStrategy respects max_parallelism settings
3. ✅ **No manual updates needed**: Never had to manually call `update_trial_data()` with fantasy values
4. ✅ **Clean workflow**: Simply generate trials, complete them when ready, and repeat

### Key Takeaways:

- **Use `get_next_trials()`** for batch generation in production workflows
- **Ax automatically tracks pending trials** based on trial status (RUNNING/STAGED)
- **Parallelism limits are enforced** by the GenerationStrategy
- **No need for manual fantasy point updates** - this is handled internally

### Migration from Manual Approach:

If you're currently using `update_trial_data()` to add fantasy points:
1. Remove those calls
2. Replace `get_next_trial()` loops with `get_next_trials()`
3. Let Ax manage trial states automatically

See `BATCH_CONDITIONING_GUIDE.md` for more details!